# 04 Repeat Purchase Driver Analysis

本 notebook 基于用户首单体验构建特征，并用一个轻量级 Logistic Regression 预测 `是否在 60 天内复购`。

说明：当前运行环境没有 `pandas / sklearn`，因此这里使用 `sqlite3 + Python standard library` 完成特征抽取、训练与解释，保证 notebook 仍然可以直接执行。


In [ ]:
from pathlib import Path
import math
import random
import sqlite3
import statistics
from collections import defaultdict

ROOT = Path.cwd()
if not (ROOT / 'sql').exists():
    ROOT = ROOT.parent
DB_PATH = ROOT / 'data' / 'raw' / 'olist.sqlite'
SEED = 42
TEST_SIZE = 0.2

if not DB_PATH.exists():
    raise FileNotFoundError(f'Missing database: {DB_PATH}')

conn = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True)
conn.row_factory = sqlite3.Row
print(f'Connected to: {DB_PATH}')


## 1. Build User-level Dataset

特征全部来自首个有效订单，标签定义为：该用户是否在首单购买后 `60 天内` 再次产生新的有效订单。

为了避免观察窗偏差，仅保留距离数据库中最后一个购买时间至少满 60 天的首单用户。


In [ ]:
DATASET_SQL = '''
WITH pay AS (
  SELECT order_id, SUM(payment_value) AS paid_value
  FROM order_payments
  GROUP BY 1
),
itm AS (
  SELECT order_id,
         SUM(price) AS items_price,
         SUM(freight_value) AS freight_value,
         SUM(price + freight_value) AS items_value,
         COUNT(*) AS item_cnt
  FROM order_items
  GROUP BY 1
),
valid_orders AS (
  SELECT
    o.order_id,
    c.customer_unique_id,
    o.order_purchase_timestamp,
    o.order_approved_at,
    o.order_delivered_carrier_date,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date,
    COALESCE(pay.paid_value, 0) AS paid_value,
    COALESCE(itm.items_value, 0) AS items_value,
    COALESCE(itm.freight_value, 0) AS freight_value,
    COALESCE(itm.item_cnt, 0) AS item_cnt
  FROM orders o
  LEFT JOIN customers c ON o.customer_id = c.customer_id
  LEFT JOIN pay ON o.order_id = pay.order_id
  LEFT JOIN itm ON o.order_id = itm.order_id
  WHERE o.order_status NOT IN ('canceled', 'unavailable')
    AND COALESCE(pay.paid_value, 0) > 0
    AND c.customer_unique_id IS NOT NULL
    AND o.order_purchase_timestamp IS NOT NULL
),
review_dedup AS (
  SELECT
    order_id,
    review_score,
    ROW_NUMBER() OVER (
      PARTITION BY order_id
      ORDER BY review_answer_timestamp DESC, review_creation_date DESC, review_id DESC
    ) AS rn
  FROM order_reviews
),
review_final AS (
  SELECT order_id, review_score
  FROM review_dedup
  WHERE rn = 1
),
order_item_features AS (
  SELECT
    oi.order_id,
    COUNT(DISTINCT p.product_category_name) AS category_diversity
  FROM order_items oi
  LEFT JOIN products p
    ON oi.product_id = p.product_id
  GROUP BY 1
),
ordered AS (
  SELECT
    vo.*,
    LEAD(vo.order_purchase_timestamp) OVER (
      PARTITION BY vo.customer_unique_id
      ORDER BY vo.order_purchase_timestamp, vo.order_id
    ) AS next_purchase_ts,
    ROW_NUMBER() OVER (
      PARTITION BY vo.customer_unique_id
      ORDER BY vo.order_purchase_timestamp, vo.order_id
    ) AS order_rank
  FROM valid_orders vo
),
max_dt AS (
  SELECT MAX(order_purchase_timestamp) AS max_purchase_ts
  FROM valid_orders
)
SELECT
  od.customer_unique_id,
  od.order_id,
  od.order_purchase_timestamp AS first_purchase_ts,
  CASE
    WHEN od.next_purchase_ts IS NOT NULL
     AND julianday(od.next_purchase_ts) <= julianday(od.order_purchase_timestamp) + 60
    THEN 1 ELSE 0 END AS repeat_60d,
  od.paid_value AS first_paid_value,
  od.items_value AS first_items_value,
  od.freight_value AS first_freight_value,
  od.item_cnt AS first_item_cnt,
  COALESCE(oif.category_diversity, 0) AS category_diversity,
  CASE
    WHEN od.order_purchase_timestamp IS NOT NULL
     AND od.order_delivered_customer_date IS NOT NULL
    THEN julianday(od.order_delivered_customer_date) - julianday(od.order_purchase_timestamp)
    ELSE NULL END AS purchase_to_delivered_days,
  CASE
    WHEN od.order_estimated_delivery_date IS NOT NULL
     AND od.order_delivered_customer_date IS NOT NULL
    THEN julianday(od.order_delivered_customer_date) - julianday(od.order_estimated_delivery_date)
    ELSE NULL END AS delivery_delay_vs_estimate_days,
  rf.review_score,
  CASE WHEN rf.review_score IS NOT NULL AND rf.review_score <= 2 THEN 1 ELSE 0 END AS is_bad_review,
  CASE
    WHEN od.order_estimated_delivery_date IS NOT NULL
     AND od.order_delivered_customer_date IS NOT NULL
     AND julianday(od.order_delivered_customer_date) - julianday(od.order_estimated_delivery_date) > 0
    THEN 1 ELSE 0 END AS is_delayed,
  CASE WHEN od.order_delivered_customer_date IS NULL THEN 1 ELSE 0 END AS missing_delivery,
  CASE WHEN rf.review_score IS NULL THEN 1 ELSE 0 END AS missing_review
FROM ordered od
LEFT JOIN order_item_features oif
  ON od.order_id = oif.order_id
LEFT JOIN review_final rf
  ON od.order_id = rf.order_id
CROSS JOIN max_dt md
WHERE od.order_rank = 1
  AND julianday(md.max_purchase_ts) - julianday(od.order_purchase_timestamp) >= 60
'''

rows = [dict(row) for row in conn.execute(DATASET_SQL)]
print('users:', len(rows))
print('60d repeat rate:', round(sum(row['repeat_60d'] for row in rows) / len(rows), 4))
rows[:2]


## 2. Prepare Features

这里保留业务上较容易解释的首单特征：

- 金额与篮子大小：`log_paid_value`, `log_item_cnt`
- 商品广度：`category_diversity`
- 物流成本结构：`freight_share`
- 履约体验：`purchase_to_delivered_days`, `delivery_delay_vs_estimate_days`, `is_delayed`
- 满意度：`review_score`, `is_bad_review`
- 缺失指示：`missing_delivery`, `missing_review`

数值型特征使用训练集 `median` 填补缺失，并按训练集均值/标准差标准化，便于直接比较系数大小。


In [ ]:
FEATURES = [
    'log_paid_value',
    'log_item_cnt',
    'category_diversity',
    'freight_share',
    'purchase_to_delivered_days',
    'delivery_delay_vs_estimate_days',
    'review_score',
    'is_delayed',
    'is_bad_review',
    'missing_delivery',
    'missing_review',
]

processed = []
for row in rows:
    items_value = row['first_items_value'] or 0.0
    freight_value = row['first_freight_value'] or 0.0
    processed.append({
        'y': float(row['repeat_60d']),
        'log_paid_value': math.log1p(row['first_paid_value'] or 0.0),
        'log_item_cnt': math.log1p(row['first_item_cnt'] or 0.0),
        'category_diversity': float(row['category_diversity'] or 0.0),
        'freight_share': float(freight_value / items_value) if items_value > 0 else 0.0,
        'purchase_to_delivered_days': row['purchase_to_delivered_days'],
        'delivery_delay_vs_estimate_days': row['delivery_delay_vs_estimate_days'],
        'review_score': row['review_score'],
        'is_delayed': float(row['is_delayed'] or 0.0),
        'is_bad_review': float(row['is_bad_review'] or 0.0),
        'missing_delivery': float(row['missing_delivery'] or 0.0),
        'missing_review': float(row['missing_review'] or 0.0),
    })

random.Random(SEED).shuffle(processed)
split_idx = int(len(processed) * (1 - TEST_SIZE))
train_rows = processed[:split_idx]
test_rows = processed[split_idx:]

feature_stats = {}
for feature in FEATURES:
    non_null_values = [row[feature] for row in train_rows if row[feature] is not None]
    median = statistics.median(non_null_values) if non_null_values else 0.0
    filled = [median if row[feature] is None else float(row[feature]) for row in train_rows]
    mean = sum(filled) / len(filled)
    variance = sum((value - mean) ** 2 for value in filled) / len(filled)
    std = math.sqrt(variance) or 1.0
    feature_stats[feature] = {'median': median, 'mean': mean, 'std': std}


def vectorize(dataset):
    X, y = [], []
    for row in dataset:
        values = [1.0]
        for feature in FEATURES:
            stats = feature_stats[feature]
            raw_value = stats['median'] if row[feature] is None else float(row[feature])
            values.append((raw_value - stats['mean']) / stats['std'])
        X.append(values)
        y.append(row['y'])
    return X, y


X_train, y_train = vectorize(train_rows)
X_test, y_test = vectorize(test_rows)
print('train size:', len(X_train), 'test size:', len(X_test))


## 3. Train Lightweight Logistic Regression

这里实现一个带 L2 正则的 batch gradient descent 版本，用于拿到方向性解释和基准预测能力。


In [ ]:
def train_logistic_regression(X, y, learning_rate=0.15, reg_lambda=0.001, epochs=120):
    weights = [0.0] * len(X[0])
    columns = list(zip(*X))

    for _ in range(epochs):
        errors = []
        for row, target in zip(X, y):
            z = sum(w * x for w, x in zip(weights, row))
            z = max(min(z, 35), -35)
            prob = 1.0 / (1.0 + math.exp(-z))
            errors.append(prob - target)

        for idx, col in enumerate(columns):
            grad = sum(err * value for err, value in zip(errors, col)) / len(X)
            if idx > 0:
                grad += reg_lambda * weights[idx]
            weights[idx] -= learning_rate * grad

    return weights


def predict_proba(X, weights):
    probabilities = []
    for row in X:
        z = sum(w * x for w, x in zip(weights, row))
        z = max(min(z, 35), -35)
        probabilities.append(1.0 / (1.0 + math.exp(-z)))
    return probabilities


def roc_auc_score(y_true, y_score):
    pairs = sorted(zip(y_score, y_true), key=lambda item: item[0])
    pos = sum(y_true)
    neg = len(y_true) - pos
    rank_sum = 0.0
    i = 0
    while i < len(pairs):
        j = i + 1
        while j < len(pairs) and pairs[j][0] == pairs[i][0]:
            j += 1
        avg_rank = (i + 1 + j) / 2.0
        rank_sum += avg_rank * sum(label for _, label in pairs[i:j])
        i = j
    return (rank_sum - pos * (pos + 1) / 2.0) / (pos * neg)


def log_loss(y_true, y_prob):
    eps = 1e-15
    total = 0.0
    for actual, pred in zip(y_true, y_prob):
        pred = min(max(pred, eps), 1 - eps)
        total += -(actual * math.log(pred) + (1 - actual) * math.log(1 - pred))
    return total / len(y_true)


weights = train_logistic_regression(X_train, y_train)
test_prob = predict_proba(X_test, weights)
print('test AUC:', round(roc_auc_score(y_test, test_prob), 4))
print('test logloss:', round(log_loss(y_test, test_prob), 4))
print('test base rate:', round(sum(y_test) / len(y_test), 4))
print('avg predicted rate:', round(sum(test_prob) / len(test_prob), 4))


## 4. Top Drivers

因为所有特征都已标准化，所以系数大小可以直接近似比较。`odds_ratio = exp(coef)` 可理解为“该特征提升 1 个标准差时，对复购 odds 的相对影响”。


In [ ]:
coefs = []
for feature, coef in zip(FEATURES, weights[1:]):
    coefs.append({
        'feature': feature,
        'coef': coef,
        'odds_ratio_per_1sd': math.exp(coef),
    })

positive = sorted(coefs, key=lambda item: item['coef'], reverse=True)[:5]
negative = sorted(coefs, key=lambda item: item['coef'])[:5]

print('Top positive drivers')
for row in positive:
    print(row['feature'], '| coef=', round(row['coef'], 4), '| odds_ratio=', round(row['odds_ratio_per_1sd'], 4))

print('\nTop negative drivers')
for row in negative:
    print(row['feature'], '| coef=', round(row['coef'], 4), '| odds_ratio=', round(row['odds_ratio_per_1sd'], 4))


## 5. Bucket-level Business Checks

模型系数只给方向，业务汇报更需要可读的分桶结果。下面补几组首单体验分桶，帮助把结论翻译成动作。


In [ ]:
def summarize_bucket(name, bucket_fn):
    agg = defaultdict(lambda: [0, 0])
    for row in rows:
        bucket = bucket_fn(row)
        agg[bucket][0] += 1
        agg[bucket][1] += row['repeat_60d']

    print(name)
    for bucket, (cnt, rep) in sorted(agg.items()):
        print(f'  {bucket:<18} users={cnt:<6} repeat_60d={rep / cnt:.2%}')
    print()


summarize_bucket(
    'item_cnt_bucket',
    lambda row: '1 item' if row['first_item_cnt'] == 1 else ('2 items' if row['first_item_cnt'] == 2 else '3+ items'),
)

summarize_bucket(
    'category_bucket',
    lambda row: '1 category' if (row['category_diversity'] or 0) <= 1 else '2+ categories',
)

summarize_bucket(
    'basket_value_bucket',
    lambda row: '<50' if row['first_paid_value'] < 50 else ('50-100' if row['first_paid_value'] < 100 else ('100-200' if row['first_paid_value'] < 200 else '200+')),
)

summarize_bucket(
    'delay_bucket',
    lambda row: 'Missing' if row['delivery_delay_vs_estimate_days'] is None else ('On-time/Early' if row['delivery_delay_vs_estimate_days'] <= 0 else ('1-3d late' if row['delivery_delay_vs_estimate_days'] <= 3 else ('4-7d late' if row['delivery_delay_vs_estimate_days'] <= 7 else '7d+ late'))),
)


## 6. Takeaways

建议重点关注两件事：

1. 真实可操作的强信号，例如首单是否形成更丰富的购物篮子。
2. 模型解释力有限这件事本身，因为它说明仅靠首单交易/履约字段还不够，需要引入更靠近用户意图的变量，例如渠道、类目、地域、卖家质量、优惠与支付方式等。
